# Обучение модели Diabetes

Домашнее задание по теме 24: Полноценный ML-сервис.

1. Загрузка датасета sklearn diabetes (scaled=False)
2. Обучение RandomForestRegressor с StandardScaler в пайплайне
3. Регистрация в MLFlow как модель "diabets"
4. Загрузка модели через mlflow.sklearn.load_model и сохранение для сервиса

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Настройка MLflow: предварительно запустите в терминале:
#   mlflow server --host 0.0.0.0 --port 5000
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("diabetes_prediction")

In [ ]:
# Загрузка датасета (scaled=False по заданию)
diabetes = load_diabetes(scaled=False)
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
print(f"Features: {feature_names}")

In [ ]:
# Пайплайн: StandardScaler + RandomForestRegressor
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42))
])

with mlflow.start_run() as run:
    # Обучение
    pipeline.fit(X_train, y_train)
    
    # Метрики
    y_pred = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)
    
    # Регистрация модели с названием "diabets"
    mlflow.sklearn.log_model(pipeline, "model", registered_model_name="diabets")
    
    print(f"Run ID: {run.info.run_id}")
    print(f"MSE: {mse:.4f}, R2: {r2:.4f}")

In [ ]:
# Загрузка модели через mlflow.sklearn.load_model
# Используем URI зарегистрированной модели (последняя версия)
model_uri = f"models:/diabets/latest"
loaded_model = mlflow.sklearn.load_model(model_uri)

# Проверка предсказания
sample = X_test[:1]
pred = loaded_model.predict(sample)
print(f"Sample prediction: {pred[0]:.2f}")
print(f"Actual: {y_test[0]:.2f}")

In [ ]:
# Сохранение модели для ML сервиса (копирование в mlapp/model)
import shutil
from pathlib import Path

# Сохраняем модель из реестра MLflow в папку mlapp для сервиса
model_dir = Path("../mlapp/model")
model_dir.mkdir(parents=True, exist_ok=True)

# Сохраняем загруженную модель в папку mlapp
mlflow.sklearn.save_model(loaded_model, str(model_dir))

print(f"Model saved to {model_dir.absolute()}")